# 12 · 학습 — **transfer** (짧은 task, horizon 축 앵커)

`transfer` = **AlohaTransferCube-v0**, `lerobot/aloha_sim_transfer_cube_human`.
`insertion`(메인)보다 **짧고 쉬운** task — 과거에 ACT 가 우리를 이겼던 판이다.

**왜 그래도 돌리나**: Intro 의 주장이 *"짧은 task 는 ACT 와 대등, horizon 이 길어질수록 우리가 앞선다"* 인데,
그 문장을 쓰려면 **짧은 쪽 데이터포인트가 있어야** 한다. transfer 는 그 앵커다.
여기서 ACT 와 대등하게만 나와도 주장이 성립하고, 지면 "짧은 task 에선 손해"라는 한계로 정직하게 적으면 된다.

출력 경로가 task 별로 갈리므로(`.../train/transfer/...` vs `.../train/insertion/...`)
insertion 결과와 **섞이지 않는다**. GPU 4개면 두 task 를 동시에는 못 돌리니 순차로.

**프로토콜은 메인과 동일**: 150k step · seed 4개 · lr 고정 · 학습중 eval OFF.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.SHORT_SIM       # ★ 'transfer' (AlohaTransferCube-v0) — 짧은 앵커
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3]
GPUS  = cf.v23.available_gpus()
NGPU  = len(GPUS)

# ── 무엇을 돌릴지 (그룹 프리셋) ────────────────────────────────────────────
TAGS = cf.GROUP_OURS + cf.GROUP_ACM     # ['ours', 'acm'] — 핵심 비교 (8잡)
# TAGS = cf.GROUP_BASELINE              #   act·diffusion·smolvla·acm2 (16잡)
# TAGS = cf.GROUP_ABLATION              #   사다리 나머지 (12잡)
# TAGS = cf.TRAIN_ALL                   #   전부 (24잡)

print('GPU  :', GPUS, f'({NGPU}개)')
print('task :', TASK, cf.v23.TASKS[TASK], '| fps', cf.fps_of(TASK))
print('학습 :', TAGS, '| seeds:', SEEDS, '| 잡:', len(TAGS) * len(SEEDS))
print('steps:', f'{cf.STEPS:,}', '| 학습중 eval:', cf.v23.EVAL_FREQ, '(0=OFF)')

## 커맨드 확인 (dry-run) — dataset/env 가 transfer 인지 볼 것

In [ ]:
for t in TAGS:
    c = cf.make_train_cmd(t, seed=SEEDS[0], task=TASK, gpu_id=0)
    print(f'{t:<10}', ' '.join(p for p in c.split()
                               if p.startswith(('--dataset.repo_id', '--env.task', '--steps',
                                                '--policy.optimizer_lr'))))
print()
print(cf.make_train_cmd(TAGS[0], seed=SEEDS[0], task=TASK, gpu_id=0))

## 학습 (resume 자동)
첫 실행은 transfer 데이터셋을 한 번 먼저 받는다(`prefetch`). 이걸 건너뛰고 잡 N개를 동시에 띄우면
같은 HF 캐시에 동시 다운로드가 걸려 대부분 죽는다.

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, ngpu=NGPU)

## 상태

In [ ]:
cf.print_training_status(jobs)
print()
cf.print_ckpt_status(TAGS, SEEDS, TASK)
print('\n다음: 13_eval_transfer')